## Instruction Finetuning

- The instructions serve as inputs for the LLMs.

- The goal for the LLM is to generate a desired response.

- Training on a dataset where the input-output pairs are explicitly provided is known as `supervised instruction finetuning`.

### Preparing and Processing Dataset

In [ ]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context=ssl.create_default_context()
    ssl_context.check_hostname=False
    ssl_context.verify_mode=ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data=response.read().decode('utf-8-sig')
        with open(file_path, 'w', encoding='utf-8-sig') as file:
            file.write(text_data)
    else:
        with open(file_path, 'r', encoding='utf-8-sig') as file:
            text_data = file.read()
    
    with open(file_path, 'r', encoding='utf-8-sig') as file:
        data=json.load(file)

    return data

file_path="instruction-data.json"
url=(
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data=download_and_load_file(file_path, url)
print(f"Number of entries: {len(data)}")

In [ ]:
print(f'Example entry: {data[50]}')

#### Converting Instructions into Alpaca Format

In [ ]:
def format_input(entry):
    instruction_text=(
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text=f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text+input_text

In [ ]:
model_input=format_input(data[50])
output_text=f"\n\n### Response:\n{data[50]['output']}"

print(model_input+output_text)

In [ ]:
print(data[999])

In [ ]:
model_input=format_input(data[999])
output_text=f"\n\n### Response:\n{data[999]['output']}"

print(model_input+output_text)

#### Splitting Dataset into Train-Val-Test Datasets

In [ ]:
train_ratio=int(len(data)*0.85)
test_ratio=int(len(data)*0.1)
val_ratio=len(data)-train_ratio-test_ratio

train_data=data[:train_ratio]
test_data=data[train_ratio:train_ratio+test_ratio]
val_data=data[train_ratio+test_ratio:]

print(f"Train dataset size: {len(train_data)}")
print(f"Validation dataset size: {len(val_data)}")
print(f"Test dataset size: {len(test_data)}")

### Organizing Data into Training Batches

- We have to batch various samples of different sizes together by converting them into `numeric representations` of same size.

- Steps to be followed:

  * `Format data using prompt template`: Format input into an instruction-response template using alpaca format.

  * `Tokenize formatted data`: Convert instruction-response entry into token IDs.

  * `Adjust to the same length with padding tokens`: Add end-of-text tokens(50256) to pad all data samples of a batch to the same length based on the longest input token ID sequence.

  * `Create target token IDs for training`: Create a list of target token IDs for the model to learn(these are the input token IDs shifted bu 1, plus an additional padding token).

  * `Replace padding tokens with placeholders`: Replace certain padding tokens by -100 to exclude them from contributing to the training loss. We dont modify the first instance of the padding token but replace all subsequent padding tokens with -100.

In [ ]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data=data

        # Pre-tokenize the text
        self.encoded_texts=[]
        for entry in data:
            instruction_plus_input=format_input(entry)
            response_text=f"\n\n### Response:\n{entry['output']}"
            full_text=instruction_plus_input+response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

In [ ]:
import tiktoken

tokenizer=tiktoken.get_encoding('gpt2')

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

`Step 1`: Find the longest sequence in the batch

`Step 2`: Pad and prepare the inputs

`Step 3`: Remove extra padded token added earlier

`Step 4`: Convert list of inputs to tensor and transfer to target device

In [ ]:
def custom_collate_1(batch, pad_token=50256, device="cpu"):
    # Find the longest sequence in the batch and increase the max length by 1, which will add one extra padding token below
    batch_max_length=max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    input_lst=[]
    for item in batch:
        new_item=item.copy()
        # Add a padding token - to easily create target tokens
        new_item+=[pad_token]
        # Pad sequences to batch_max_length
        padded=(
            new_item+[pad_token]*(batch_max_length-len(new_item))
        )
        # With padded[:-1], remove the extra padded token added earlier
        inputs=torch.tensor(padded[:-1])
        input_lst.append(inputs)

    # Convert list of inputs to tensor and transfer to target device
    input_tensor=torch.stack(input_lst).to(device)
    return input_tensor

In [ ]:
input_1=[0, 1, 2, 3, 4]
input_2=[5, 6]
input_3=[7, 8, 9]

batch=(
    input_1,
    input_2,
    input_3
)

print(custom_collate_1(batch))

#### Creating Target Token IDs

In [ ]:
def custom_collate_2(batch, pad_token=50256, device="cpu"):
    # Find the longest sequence in the batch and increase the max length by 1, which will add one extra padding token below
    batch_max_length=max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    input_lst, target_lst=[], []
    for item in batch:
        new_item=item.copy()
        # Add a padding token - to easily create target tokens
        new_item+=[pad_token]
        # Pad sequences to batch_max_length
        padded=(
            new_item+[pad_token]*(batch_max_length-len(new_item))
        )
        # With padded[:-1], remove the extra padded token added earlier
        inputs=torch.tensor(padded[:-1])
        targets=torch.tensor(padded[1:])
        input_lst.append(inputs)
        target_lst.append(targets)

    # Convert list of inputs to tensor and transfer to target device
    input_tensor=torch.stack(input_lst).to(device)
    target_tensor=torch.stack(target_lst).to(device)
    return input_tensor, target_tensor

In [ ]:
inputs, targets=custom_collate_2(batch)

print(f"Inputs:\n{inputs}\n")
print(f"Targets:\n{targets}")

In [ ]:
def custom_collate_final(batch, pad_token=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    # Find the longest sequence in the batch and increase the max length by 1, which will add one extra padding token below
    batch_max_length=max(len(item)+1 for item in batch)

    # Pad and prepare inputs
    input_lst, target_lst=[], []
    for item in batch:
        new_item=item.copy()
        # Add a padding token - to easily create target tokens
        new_item+=[pad_token]
        # Pad sequences to batch_max_length
        padded=(
            new_item+[pad_token]*(batch_max_length-len(new_item))
        )
        # With padded[:-1], remove the extra padded token added earlier
        inputs=torch.tensor(padded[:-1])
        targets=torch.tensor(padded[1:])

        # Replace all but the first padding tokens in targets
        mask=targets==pad_token
        indices=torch.nonzero(mask).squeeze()
        if indices.numel()>1:
            targets[indices[1:]]=ignore_index

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs=inputs[:allowed_max_length]
            targets=targets[:allowed_max_length]

        input_lst.append(inputs)
        target_lst.append(targets)

    # Convert list of inputs to tensor and transfer to target device
    input_tensor=torch.stack(input_lst).to(device)
    target_tensor=torch.stack(target_lst).to(device)
    
    return input_tensor, target_tensor

In [ ]:
inputs, targets=custom_collate_final(batch)

print(f"Inputs:\n{inputs}\n")
print(f"Targets:\n{targets}")

#### ✅ Example Walkthrough

Suppose we have a sequence:
`[11, 22, 33, 50256, 50256, 50256]`

After this code:

* `mask = [False, False, False, True, True, True]`
* `indices = [[3], [4], [5]]` - before .squeeze()
* `indices = [3, 4, 5]` - after .squeeze()
* `targets[indices[1:]] = -100`

Final `targets`:
`[11, 22, 33, 50256, -100, -100]`

So only the **first pad** remains, others are ignored.

In [ ]:
logits_1=torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1=torch.tensor([0, 1])

loss_1=torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

In [ ]:
logits_2=torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5],
     [-0.5, 1.5]],
)
targets_2=torch.tensor([0, 1, 1])

loss_2=torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

In [ ]:
logits_2=torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5],
     [-0.5, 1.5]],
)
targets_3=torch.tensor([0, 1, -100])

# Since the 3rd target value is -100, there will be no difference in loss when we add the 3rd training example
loss_3=torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print(f'loss_1==loss_3: {loss_1==loss_3}')

#### Masking Target Token IDs

- In addition to masking out padding tokens, it is also common to mask out the target token IDs that correspond to the instruction.

- By doing this, LLM cross entropy loss is only computed for the generated response target IDs.

- By masking out the instruction target token IDs, we ensure that the model learns to generate accurate responses rather than memorizing instructions additionally, which can help to reduce overfitting.

### DataLoaders for Instruction Finetuning

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

In [ ]:
from functools import partial
customized_collate_fn=partial(custom_collate_final, device=device, allowed_max_length=1024)

In [ ]:
from torch.utils.data import DataLoader

num_workers=0
batch_size=8

train_dataset=InstructionDataset(train_data, tokenizer)
train_dataloader=DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=customized_collate_fn,
    drop_last=True,
    num_workers=num_workers
)

val_dataset=InstructionDataset(val_data, tokenizer)
val_dataloader=DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=customized_collate_fn,
    drop_last=True,
    num_workers=num_workers
)

test_dataset=InstructionDataset(test_data, tokenizer)
test_dataloader=DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=customized_collate_fn,
    drop_last=False,
    num_workers=num_workers
)

In [ ]:
print(f"Train DataLoader size: {len(train_dataloader)}")

for input, target in train_dataloader:
    print(input.shape, target.shape)

### Loading Pre-Trained Weights into LLM

In [ ]:
CHOOSE_MODEL="gpt2-small (124M)"

BASE_CONFIG={
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs={
  "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
  "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
  "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
  "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [ ]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len=in_idx.shape
    tok_embeds=self.tok_emb(in_idx)
    pos_embeds=self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x=tok_embeds+pos_embeds
    x=self.drop_emb(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits
  
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      dropout=cfg["drop_rate"],
      num_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut=x
    # Every row in the input has 0 mean and 1 variance
    x=self.norm1(x)
    # We get the context vector of [batch_size, num_tokens, emb_dim]
    x=self.att(x)
    # Dropout layer to improve efficiency
    x=self.drop_shortcut(x)
    # Creating shortcut connection
    x=x+shortcut

    # Shortcut for feed forward block
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    return x
  
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs
  
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift
  
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu
  
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)
  

import numpy as np

def load_weights_into_gpt(gpt, params):
  gpt.pos_emb.weight=assign(gpt.pos_emb.weight, params["wpe"])
  gpt.tok_emb.weight=assign(gpt.tok_emb.weight, params["wte"])

  for b in range(len(params["blocks"])):
    q_w, k_w, v_w=np.split(
      params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.weight=assign(
      gpt.trf_blocks[b].att.w_query.weight, q_w.T
    )
    gpt.trf_blocks[b].att.w_key.weight=assign(
      gpt.trf_blocks[b].att.w_key.weight, k_w.T
    )
    gpt.trf_blocks[b].att.w_value.weight=assign(
      gpt.trf_blocks[b].att.w_value.weight, v_w.T
    )

    q_b, k_b, v_b=np.split(
      params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.bias=assign(
      gpt.trf_blocks[b].att.w_query.bias, q_b
    )
    gpt.trf_blocks[b].att.w_key.bias=assign(
      gpt.trf_blocks[b].att.w_key.bias, k_b
    )
    gpt.trf_blocks[b].att.w_value.bias=assign(
      gpt.trf_blocks[b].att.w_value.bias, v_b
    )

    gpt.trf_blocks[b].att.out_proj.weight=assign(
      gpt.trf_blocks[b].att.out_proj.weight,
      params["blocks"][b]["attn"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].att.out_proj.bias=assign(
      gpt.trf_blocks[b].att.out_proj.bias,
      params["blocks"][b]["attn"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].ff.layers[0].weight=assign(
      gpt.trf_blocks[b].ff.layers[0].weight,
      params["blocks"][b]["mlp"]["c_fc"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[0].bias=assign(
      gpt.trf_blocks[b].ff.layers[0].bias,
      params["blocks"][b]["mlp"]["c_fc"]["b"]
    )
    gpt.trf_blocks[b].ff.layers[2].weight=assign(
      gpt.trf_blocks[b].ff.layers[2].weight,
      params["blocks"][b]["mlp"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[2].bias=assign(
      gpt.trf_blocks[b].ff.layers[2].bias,
      params["blocks"][b]["mlp"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].norm1.scale=assign(
      gpt.trf_blocks[b].norm1.scale,
      params["blocks"][b]["ln_1"]["g"]
    )
    gpt.trf_blocks[b].norm1.shift=assign(
      gpt.trf_blocks[b].norm1.shift,
      params["blocks"][b]["ln_1"]["b"]
    )
    gpt.trf_blocks[b].norm2.scale=assign(
      gpt.trf_blocks[b].norm2.scale,
      params["blocks"][b]["ln_2"]["g"]
    )
    gpt.trf_blocks[b].norm2.shift=assign(
      gpt.trf_blocks[b].norm2.shift,
      params["blocks"][b]["ln_2"]["b"]
    )

  gpt.final_norm.scale=assign(gpt.final_norm.scale, params["g"])
  gpt.final_norm.shift=assign(gpt.final_norm.shift, params["b"])
  gpt.out_head.weight=assign(gpt.out_head.weight, params["wte"])

def assign(left, right):
  if left.shape!=right.shape:
    raise ValueError(f'Shape mismatch. Left Shape: {left.shape}, Right Shape: {right.shape}')
  return torch.nn.Parameter(torch.tensor(right, dtype=left.dtype, device=left.device))


def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx is (batch_size, num_tokens) array of indices in current context
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    '''For example, 
    Case 1: If LLM supports only 5 tokens and context size is
    10, then only the last 5 tokens are used as context.
    Case 2: If LLM supports 8 tokens and context size is 5, then
    only the last 5 tokens are used as context.'''
    idx_cond=idx[:, -context_size:]

    # Get output tensors - (batch_size, num_tokens, vocab_size)
    with torch.no_grad():
      logits=model(idx_cond)

    # Extract last vector
    logits=logits[:, -1, :]

    # Apply softmax to get probabilities - (batch_size, vocab_size)
    probs=torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next=torch.argmax(probs, dim=-1, keepdim=True)
    # (batch_size, 1)

    # Append sampled index to the running sequence
    idx=torch.cat((idx, idx_next), dim=1)
    # (batch_size, num_tokens+1)

  return idx

def text_to_token_ids(text, tokenizer):
  encoded_text=tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded_text).unsqueeze(0)
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat=token_ids.squeeze(0)
  decoded_text=tokenizer.decode(flat.tolist())
  return decoded_text

In [ ]:
model_size=CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

from gpt_download import download_and_load_gpt2

settings, params=download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model=GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
torch.manual_seed(123)
input_text=format_input(val_data[0])
print(input_text)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temp=0.0, top_k=None, eos_id=None):
  for _ in range(max_new_tokens):
    idx_cond=idx[:, -context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:, -1, :]

    # Filter logits with top-k sampling
    if top_k is not None:
      top_k_logits, _=torch.topk(logits, top_k)
      min_val=top_k_logits[:, -1]
      logits=torch.where(
        condition=logits<min_val,
        input=torch.tensor(float("-inf")).to(device),
        other=logits
      )

    # Apply temperature scaling
    if temp>0.0:
      logits=logits/temp
      probs=torch.softmax(logits, dim=-1)
      idx_next=torch.multinomial(probs, num_samples=1)
    else:
      probs=torch.softmax(logits, dim=-1)
      idx_next=torch.argmax(probs, dim=-1, keepdim=True)

    # Stop generating early if end-of-sequence token is encountered
    if idx_next==eos_id:
      break
      
    idx=torch.cat((idx, idx_next), dim=1)

  return idx

In [ ]:
token_ids=generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

generated_text=token_ids_to_text(token_ids, tokenizer)
print(generated_text)

In [ ]:
response_text=generated_text[len(input_text):].strip()
print(response_text)

### Instruction Finetuning the LLM

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch=input_batch.to(device), target_batch.to(device)
    logits=model(input_batch)
    loss=torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    model.eval()
    total_loss=0.0

    if num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i<num_batches:
            batch_loss=calc_loss_batch(input_batch, target_batch, model, device)
            total_loss+=batch_loss.item()
        else:
            break
        
    return total_loss/num_batches

In [ ]:
def train_model_sample(model, train_loader, val_loader, optimizer,
                       device, num_epochs, eval_freq, eval_iter,
                       start_context, tokenizer):
  # Initialize lists to track losses and tokens seen
  train_losses, val_losses, track_tokens_seen=[], [], []
  tokens_seen, global_step=0, -1

  # Main training loop
  for epoch in range(num_epochs):
    # Set model to training mode
    model.train()

    for input_batch, target_batch in train_loader:
      # Reset loss gradients from previous batch iteration
      optimizer.zero_grad()
      loss=calc_loss_batch(input_batch, target_batch, model, device)
      # Calculate loss gradients
      loss.backward()
      # Update model weights using loss gradients
      optimizer.step()
      # Return the total number of elements in the input batch
      tokens_seen+=input_batch.numel()
      global_step+=1

      # Optional evaluation step
      if global_step%eval_freq==0:
        train_loss, val_loss=evaluate_model(
          model, train_loader, val_loader, device, eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f'Epoch {epoch+1} (Step {global_step:06d}): '
              f'Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}')
        
    generate_and_print_sample(
      model, tokenizer, device, start_context
    )

  return train_losses, val_losses, track_tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
    val_loss=calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
  model.train()
  return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
  model.eval()
  context_size=model.pos_emb.weight.shape[0]
  encoded=text_to_token_ids(start_context, tokenizer).to(device)
  with torch.no_grad():
    token_ids=generate_text_simple(
      model, encoded, max_new_tokens=50, context_size=context_size
    )
  decoded=token_ids_to_text(token_ids, tokenizer)
  print(decoded.replace('\n', " "))
  model.train()

In [ ]:
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss=calc_loss_loader(train_dataloader, model, device, num_batches=5)
    val_loss=calc_loss_loader(val_dataloader, model, device, num_batches=5)

print(f'Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}')

In [ ]:
import time

start_time=time.time()
torch.manual_seed(123)

# weight_decay prevents overfitting - keeps track of gradients and prevents local minima, accelerates convergence
optimizer=torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs=2

train_losses, val_losses, tokens_seen=train_model_sample(
  model=model,
  train_loader=train_dataloader,
  val_loader=val_dataloader,
  optimizer=optimizer,
  device=device,
  num_epochs=num_epochs,
  eval_freq=5,
  eval_iter=5,
  start_context=format_input(val_data[0]),
  tokenizer=tokenizer
)

end_time=time.time()
exec_time_min=(end_time-start_time)/60
print(f'Training completed in {exec_time_min:.2f} minutes.')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
  fig, ax1=plt.subplots(figsize=(5, 3))

  # Plot training and validation loss against epochs
  ax1.plot(epochs_seen, train_losses, label="Training loss")
  ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
  ax1.set_xlabel("Epochs")
  ax1.set_ylabel("Loss")
  ax1.legend(loc="upper right")
  # Show only integer labels on x axis
  ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

  # Plot training and validation loss against tokens seen
  ax2=ax1.twiny()
  ax2.plot(tokens_seen, train_losses, alpha=0)
  ax2.set_xlabel("Tokens")

  fig.tight_layout()
  plt.show()

epochs_tensor=torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

### Extracting and Saving Responses in LLM

In [ ]:
torch.manual_seed(123)

for entry in test_data[:3]:
    input_text=format_input(entry)

    token_ids=generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )

    generated_text=token_ids_to_text(token_ids, tokenizer)
    response_text=(
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(f"{input_text}\n")
    print(f"Correct Response:\n>> {entry['output']}\n")
    print(f"Generated Response:\n>> {response_text.strip()}")
    print("------")

In [ ]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text=format_input(entry)

    token_ids=generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )

    generated_text=token_ids_to_text(token_ids, tokenizer)
    response_text=(
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    test_data[i]["model_response"]=response_text

with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)

In [ ]:
print(test_data[0])

In [ ]:
import re

file_name=f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"Model saved as {file_name}")

# To load model
# model.load_state_dict(torch.load(file_name))

### Evaluating the Fine-tuned LLM with Ollama

#### Evaluation Metric - MMLU(Measuring Massive Multitask Language Understanding)

- This is designed to evaluate a text model's ability to learn and apply knowledge from various domains.

- MMLU consists of 57 tasks covering STEM, humanities, social sciences and other areas ranging from elementary to professional levels of difficulty.

- This primarily uses zero-shot and few-shot settings to evaluate how well models can generalize their pre-training knowledge.

- We can use MMLU to evaluate our custom instruction fine-tuned LLM by testing its performance on the benchmark's 57 tasks.

- `Ollama` can be thought of an efficient application to run large language models locally.

- Ollama is only a tool for generating text using LLMs and does not support training or fine-tuning LLMs.

- Make sure to run the command `ollama run llama3` in the command prompt to start the model server before running the code below.

In [ ]:
import psutil

def check_if_running(process_name):
    running=False
    for proc in psutil.process_iter(['name']):
        if process_name in proc.info['name']:
            running=True
            break
    return running

ollama_running=check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama is not running")
print("Ollama is running")

In [ ]:
# Code to send the prompt directly to llama3 and get the response
import urllib

def query_model(
    prompt,
    model="llama3",
    url="http://localhost:11434/api/chat"
):
    data={
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Convert the dictionary to a JSON formatted string and encode it to bytes
    payload=json.dumps(data).encode('utf-8')

    # Create a request object, setting the method to POST and adding necessary headers
    request=urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )

    request.add_header("Content-Type", "application/json")

    # Send the request and read the response
    response_data=""
    with urllib.request.urlopen(request) as response:
        while True:
            line=response.readline().decode("utf-8")
            if not line:
                break
            response_json=json.loads(line)
            response_data+=response_json["message"]["content"]

    return response_data

In [ ]:
model="llama3"
result=query_model("What is the capital of France?", model=model)
print(result)

In [ ]:
for entry in test_data[:3]:
    prompt=(
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )

    print("\nDataset response:")
    print(">>", entry["output"])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("------\n")

In [ ]:
for entry in test_data[:3]:
    prompt=(
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
        f"Respond with the integer number only."
    )

    print("\nDataset response:")
    print(">>", entry["output"])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("------\n")

In [ ]:
def generate_model_scores(json_data, json_key, model="llama3"):
    scores=[]
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt=(
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry['model_response']}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score=query_model(prompt)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue
        
    return scores

### Improving the Fine-tuned LLM

To furthur improve our model's performance, we can explore various strategies, such as:

- Adjusting the hyperparameters during finetuning, such as learning rate, batch size, number of epochs.

- Increasing the size of training dataset or diversifying the examples to cover a broader range of topics and styles.

- Experimenting with different prompts or instruction formats to guide the model's responses more effectively.

- We can also use PEFT technique like `LoRA`.

- Considering the use of a larger pretrained model, which may have a greater capacity to capture complex patterns and generate more accurate responses.